# My personal notes on LNCC Santos Dumont (SD) supercomputer

## SD Connection
- connects to SD via SSH
- loads the Intel module
    - source /scratch/app/modulos/intel-psxe-2019.sh
- enter the working directory
    - cd /prj/....
- activates the conda environment
    - conda activate --stack ./env
- activates the Jupyter Notebook
    - jupyter notebook --no-browser --port=8889
- on the local machine:
    - ssh -NfL 8889:[::1]:8889 nnn@login.sdumont.lncc.br
    - sshfs nnn@login.sdumont.lncc.br:/ /mnt/sd

## Jupyter Notebook (JN)
There is an Anaconda distribution at /scratch/app/anaconda3/2018.12. The first step is to configure PATH, which can be done in several ways, one of which is to add it at the end of .bashrc :

```bash
# >>> conda initialize >>>
# !! Contents within this block are managed by 'conda init' !!
__conda_setup="$('/scratch/app/anaconda3/2018.12/bin/conda' 'shell.bash' 'hook' 2> /dev/null)"
if [ $? -eq 0 ]; then
    eval "$__conda_setup"
else
    if [ -f "/scratch/app/anaconda3/2018.12/etc/profile.d/conda.sh" ]; then
        . "/scratch/app/anaconda3/2018.12/etc/profile.d/conda.sh"
    else
        export PATH="/scratch/app/anaconda3/2018.12/bin:$PATH"
    fi
fi
unset __conda_setup
# <<< conda initialize <<<
```
and to load, or log out / login, or execute `source .basrc` .

Another way is by using conda:

```bash
/scratch/app/anaconda3/2018.12/bin/conda init bash
```

This way, every time you log in, the base environment is automatically configured. The active environment appears on the command line. To list the available packages, you can use `conda list`, and to list the available environments` conda env list`. To activate an environment, for example `tf_gpu`, use` conda activate tf_gpu`.

The next step now is to configure the Jupyter Notebook (JN). For this we will create a stacked local base, so new packages are in `/ prj` and the existing packages are unchanged in` / scratch`. This way both bases are used at the same time, saving space and downloading. To create the local database, enter the desired local directory (you can have several environments in different directories), and to create, for example, the environment `envs`, use` conda create --prefix ./envs `. Then, to activate, use `conda activate --stack. / Envs`.

A directory will appear in `/prj`, or wherever it is created, called `envs`, which cannot be deleted or moved.
Once the new environment is activated, as the `/scratch` JN is out of date, it is necessary to install the new updated version with:

    conda install jupyter

which will be installed on the new `envs` base. To run Fortran on JN, the `fortran-magic` extension is required:

    conda install -y -c conda-forge fortran-magic

To run the JN server on SDumont:

    jupyter notebook --no-browser --port=8889

Port 8889 can be changed. It shows the link with the token to be used in the browser. The next step now is to configure a tunnel on the local machine (laptop), where `xxxx` is the username:

    ssh -NL 8889:[::1]:8889 xxxx@login.sdumont.lncc.br

If there is an error, a possible solution is to turn off GSSAPI. Or, change the local port for 8890, for example.

Now, finally, just open the browser and insert the link with the token, and the JN will appear. Everything that is done on the JN interface on the local machine, will be performed on the SDumont login node.

## Miscellaneous commands

In [8]:
conda env list    # or "conda info --envs"

# conda environments:
#
                      *  /prj/yyyy/xxxx/stnc/envs
base                     /scratch/app/anaconda3/2018.12
tf_gpu                   /scratch/app/anaconda3/2018.12/envs/tf_gpu


Note: you may need to restart the kernel to use updated packages.


In [3]:
conda list

# packages in environment at /prj/yyyy/xxxx/stnc/envs:
#
# Name                    Version                   Build  Channel
_libgcc_mutex             0.1                        main  
attrs                     19.3.0                     py_0  
backcall                  0.1.0                    py37_0  
bleach                    3.1.4                      py_0  
ca-certificates           2020.4.5.2           hecda079_0    conda-forge
certifi                   2020.4.5.2       py37hc8dfbb8_0    conda-forge
dbus                      1.13.14              hb2f20db_0  
decorator                 4.4.2                      py_0  
defusedxml                0.6.0                      py_0  
entrypoints               0.3                      py37_0  
expat                     2.2.6                he6710b0_0  
fontconfig                2.13.0               h9420a91_0  
fortran-magic             0.7                     py_1001    conda-forge
freetype                  2.9.1                h8a8886c_1

In [5]:
%%bash
uname -a
echo
tail -n 26 /proc/cpuinfo

Linux sdumont11 3.10.0-957.el7.x86_64 #1 SMP Thu Oct 4 20:48:51 UTC 2018 x86_64 x86_64 x86_64 GNU/Linux

processor	: 23
vendor_id	: GenuineIntel
cpu family	: 6
model		: 62
model name	: Intel(R) Xeon(R) CPU E5-2695 v2 @ 2.40GHz
stepping	: 4
microcode	: 0x42d
cpu MHz		: 3000.000
cache size	: 30720 KB
physical id	: 1
siblings	: 12
core id		: 13
cpu cores	: 12
apicid		: 58
initial apicid	: 58
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush dts acpi mmx fxsr sse sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm constant_tsc arch_perfmon pebs bts rep_good nopl xtopology nonstop_tsc aperfmperf eagerfpu pni pclmulqdq dtes64 monitor ds_cpl vmx smx est tm2 ssse3 cx16 xtpr pdcm pcid dca sse4_1 sse4_2 x2apic popcnt tsc_deadline_timer aes xsave avx f16c rdrand lahf_lm epb ssbd ibrs ibpb stibp tpr_shadow vnmi flexpriority ept vpid fsgsbase smep erms xsaveopt dtherm ida arat pln pts spec_ctrl intel_stibp f

In [1]:
%load_ext fortranmagic

In [10]:
%%fortran -vvv
subroutine my_function(x, y, z)
    real, intent(in) :: x(:), y(:)
    real, intent(out) :: z(size(x))
    ! using vector operations
    z(:) = sin(x(:) + y(:))
end subroutine

Running...
   /prj/yyyy/xxxx/stnc/envs/bin/python -m numpy.f2py -m _fortran_magic_507dcd91cc688596e163c08dab6d308b -c /prj/yyyy/xxxx/.cache/ipython/fortran/_fortran_magic_507dcd91cc688596e163c08dab6d308b.f90
running build
running config_cc
unifing config_cc, config, build_clib, build_ext, build commands --compiler options
running config_fc
unifing config_fc, config, build_clib, build_ext, build commands --fcompiler options
running build_src
build_src
building extension "_fortran_magic_507dcd91cc688596e163c08dab6d308b" sources
f2py options: []
f2py:> /tmp/tmpk4emh5_b/src.linux-x86_64-3.7/_fortran_magic_507dcd91cc688596e163c08dab6d308bmodule.c
creating /tmp/tmpk4emh5_b/src.linux-x86_64-3.7
Reading fortran codes...
	Reading file '/prj/yyyy/xxxx/.cache/ipython/fortran/_fortran_magic_507dcd91cc688596e163c08dab6d308b.f90' (format:free)
Post-processing...
	Block: _fortran_magic_507dcd91cc688596e163c08dab6d308b
			Block: my_function
Post-processing (stage 2)...
Building modules...
	Building mo

In file included from /prj/yyyy/xxxx/stnc/envs/lib/python3.7/site-packages/numpy/core/include/numpy/ndarraytypes.h:1832:0,
                 from /prj/yyyy/xxxx/stnc/envs/lib/python3.7/site-packages/numpy/core/include/numpy/ndarrayobject.h:12,
                 from /prj/yyyy/xxxx/stnc/envs/lib/python3.7/site-packages/numpy/core/include/numpy/arrayobject.h:4,
                 from /tmp/tmpk4emh5_b/src.linux-x86_64-3.7/fortranobject.h:13,
                 from /tmp/tmpk4emh5_b/src.linux-x86_64-3.7/_fortran_magic_507dcd91cc688596e163c08dab6d308bmodule.c:16:
/prj/yyyy/xxxx/stnc/envs/lib/python3.7/site-packages/numpy/core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" [-Wcpp]
 #warning "Using deprecated NumPy API, disable it with " \
  ^
In file included from /prj/yyyy/xxxx/stnc/envs/lib/python3.7/site-packages/numpy/core/include/numpy/ndarraytypes.h:1832:0,
                 fro


Ok. The following fortran objects are ready to use: my_function


fortran-magic docs: https://nbviewer.jupyter.org/github/mgaitan/fortran_magic/blob/master/documentation.ipynb

In [1]:
!source /scratch/app/modulos/intel-psxe-2019.sh

Intel(R) Parallel Studio XE 2019 Update 3 for Linux*
Copyright (C) 2009-2019 Intel Corporation. All rights reserved.
/opt/intel/parallel_studio_xe_2019/intelpython3/etc/conda/deactivate.d/mpivars.deactivate.sh: line 37: syntax error near unexpected token `('
/opt/intel/parallel_studio_xe_2019/intelpython3/etc/conda/deactivate.d/mpivars.deactivate.sh: line 37: `if grep -q "compilers_and_libraries.*mpi" <(echo $FIP); then'


In [2]:
%reload_ext fortranmagic

In [3]:
%%fortran -v --fcompiler=intelem --opt '-O3'
subroutine my_function(x, y, z)
    real, intent(in) :: x(:), y(:)
    real, intent(out) :: z(size(x))
    ! using vector operations
    z(:) = sin(x(:) + y(:))
end subroutine


Ok. The following fortran objects are ready to use: my_function


In [4]:
import numpy
x = numpy.random.normal(size=10)
y = numpy.random.normal(size=10)
z = my_function(x, y)
z

array([-0.96598935,  0.07206809,  0.9913887 ,  0.52801114, -0.26262093,
       -0.9999999 , -0.15687317,  0.9964671 ,  0.9969604 ,  0.3341872 ],
      dtype=float32)

In [6]:
!sinfo -p nvidia_dev -o %C

CPUS(A/I/O/T)
4317/267/24/4608


A =  number of allocated cores

I =  number of idle cores

O =  number of 'other' cores, i.e. drained, down, etc.

T =  total number of cores in the system

In [7]:
!sinfo -p nvidia -o %C

CPUS(A/I/O/T)
4317/267/24/4608


In [9]:
!sinfo -p nvidia_long -o %C

CPUS(A/I/O/T)
4317/267/24/4608


In [12]:
!sinfo -p sequana_dev -o %C

CPUS(A/I/O/T)


In [13]:
!sinfo -p sequana_gpu -o %C

CPUS(A/I/O/T)
2064/432/0/2496


In [14]:
!sinfo -p sequana_long -o %C

CPUS(A/I/O/T)


In [15]:
!sinfo -p sd_gpu -o %C

CPUS(A/I/O/T)
2304/432/48/2784


In [16]:
!sinfo -p sd_gpu_dev -o %C

CPUS(A/I/O/T)


In [18]:
!sinfo -p sd_gpu_long -o %C

CPUS(A/I/O/T)


In [19]:
!squeue -n xxxx

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


## Using

Ref.: https://sdumont.lncc.br/support_manual.php?pg=support#7.2  (in Portuguese)

In [2]:
!sacctmgr -s show user xxxx

      User   Def Acct     Admin    Cluster    Account  Partition     Share MaxJobs MaxNodes  MaxCPUs MaxSubmit     MaxWall  MaxCPUMins                  QOS   Def QOS 
---------- ---------- --------- ---------- ---------- ---------- --------- ------- -------- -------- --------- ----------- ----------- -------------------- --------- 
xxxx.m+     yyyy      None    sdumont     yyyy        gdl         1       1       50     1200         6  2-00:00:00                           normal           
xxxx.m+     yyyy      None    sdumont     yyyy     mesca2         1       1        1      480         6  2-00:00:00                           normal           
xxxx.m+     yyyy      None    sdumont     yyyy nvidia_sm+         1       4       20      480        24    01:00:00                           normal           
xxxx.m+     yyyy      None    sdumont     yyyy nvidia_lo+         1       1       10      240         1 31-00:00:00                           normal           
xxxx.m+     yyyy      None

List of jobs that were executed by the project in a given period:

In [21]:
!sacct -S 2020-01-01 -X -A yyyy

       JobID    JobName  Partition    Account  AllocCPUS      State ExitCode 
------------ ---------- ---------- ---------- ---------- ---------- -------- 
491632              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491645              sp1    cpu_dev     yyyy          4     FAILED      1:0 
491649              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491652              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491655              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491656              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491657              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491670              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491671              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491673              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491762              sp1    cpu_dev     yyyy         24  COMPLETED      0:0 
491768  

In [23]:
!sacct -j 519449

       JobID    JobName  Partition    Account  AllocCPUS      State ExitCode 
------------ ---------- ---------- ---------- ---------- ---------- -------- 
519449          stencil    cpu_dev     yyyy         48  COMPLETED      0:0 
519449.batch      batch                yyyy         24  COMPLETED      0:0 
519449.0     stencil_m+                yyyy         25  COMPLETED      0:0 
519449.1     stencil_m+                yyyy         25  COMPLETED      0:0 
519449.2     stencil_m+                yyyy         25  COMPLETED      0:0 


In [25]:
!sreport -t hours job sizesbyaccount start=2020-01-01 users=xxxx flatview grouping=12001

--------------------------------------------------------------------------------
Job Sizes 2020-01-01T00:00:00 - 2020-06-23T23:59:59 (15120000 secs)
Time reported in Hours
--------------------------------------------------------------------------------
  Cluster   Account  0-12000 CPUs >= 12001 CPUs % of cluster 
--------- --------- ------------- ------------- ------------ 
  sdumont    yyyy            10             0      100.00% 


Number of hours used by the project during a period:

In [26]:
!sreport -t hours cluster AccountUtilizationByUser start=2020-01-01 Accounts=yyyy

--------------------------------------------------------------------------------
Cluster/Account/User Utilization 2020-01-01T00:00:00 - 2020-06-23T23:59:59 (15120000 secs)
Use reported in TRES Hours
--------------------------------------------------------------------------------
  Cluster         Account     Login     Proper Name      Used   Energy 
--------- --------------- --------- --------------- --------- -------- 
  sdumont          yyyy                              108871        0 
  sdumont          yyyy aaaa+ aaaa aaaa+         0        0 
  sdumont          yyyy bbbb+ bbbb bbbb+         1        0 
  sdumont          yyyy bbbb+ bbbb bbbb+        10        0 
  sdumont          yyyy cccc+ cccc cccc+     77248        0 
  sdumont          yyyy cccc+ cccc cccc+       364        0 
  sdumont          yyyy cccc+ cccc cccc+        25        0 
  sdumont          yyyy cccc+ cccc cccc+        28        0 
  sdumont          yyyy cccc+ cccc cccc+     15860        0 
  sdumont         